In [1]:
import numpy as np
import pandas as pd
import gc
import warnings
from itertools import combinations

from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from scipy.stats import rankdata

warnings.filterwarnings("ignore")

In [2]:
# LOAD DATA
train = pd.read_csv("/kaggle/input/competitions/playground-series-s6e2/train.csv")
test  = pd.read_csv("/kaggle/input/competitions/playground-series-s6e2/test.csv")

In [3]:
# DATA PREPARATION
TARGET = "Heart Disease"
train[TARGET] = train[TARGET].map({"Absence":0, "Presence":1}).astype(np.uint8)
y = train[TARGET].values
test_ids = test["id"]

X_tr_raw = train.drop(columns=[TARGET, "id"])
X_te_raw = test.drop(columns=["id"])

cat_cols = [
    'Sex','Chest pain type','FBS over 120','EKG results',
    'Exercise angina','Slope of ST',
    'Number of vessels fluro','Thallium'
]
num_cols = ['Age','BP','Cholesterol','Max HR','ST depression']

In [4]:
# FREQUENCY ENCODING
def freq_encode(tr, te, cols):
    all_df = pd.concat([tr, te])
    tr_out, te_out = pd.DataFrame(index=tr.index), pd.DataFrame(index=te.index)
    for c in cols:
        freq = all_df[c].value_counts(normalize=True)
        tr_out[c+"_freq"] = tr[c].map(freq)
        te_out[c+"_freq"] = te[c].map(freq)
    return tr_out, te_out

tr_freq, te_freq = freq_encode(X_tr_raw, X_te_raw, cat_cols + num_cols)

In [5]:
# TARGET ENCODING (K-FOLD SAFE)
skf_te = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
tr_te = pd.DataFrame(index=X_tr_raw.index)
te_te = pd.DataFrame(index=X_te_raw.index)

for c in cat_cols + num_cols:
    tr_te[c+"_te"] = 0.0
    for tr_i, val_i in skf_te.split(X_tr_raw, y):
        means = train.iloc[tr_i].groupby(c)[TARGET].mean()
        tr_te.iloc[val_i, tr_te.columns.get_loc(c+"_te")] = X_tr_raw.iloc[val_i][c].map(means)
    te_te[c+"_te"] = X_te_raw[c].map(train.groupby(c)[TARGET].mean())

In [6]:
# FEATURE-GROWTH (APRIORI-LITE)
base = tr_te.fillna(0)
corr_scores = {}

for a, b in combinations(base.columns, 2):
    corr_scores[(a,b)] = abs(np.corrcoef(base[a], base[b])[0,1])

top_pairs = sorted(corr_scores, key=corr_scores.get, reverse=True)[:8]

for a,b in top_pairs:
    tr_te[f"{a}_x_{b}"] = tr_te[a] * tr_te[b]
    te_te[f"{a}_x_{b}"] = te_te[a] * te_te[b]

In [7]:
# FINAL FEATURE SET
X_train = pd.concat([tr_freq, tr_te], axis=1).fillna(0)
X_test  = pd.concat([te_freq, te_te], axis=1).fillna(0)

In [8]:
# GPU MODEL CONFIGS
cat_params = {
    "iterations": 10000,
    "learning_rate": 0.01,
    "depth": 2,
    "loss_function": "Logloss",
    "eval_metric": "AUC",
    "auto_class_weights": "Balanced",
    "subsample": 0.65,
    "l2_leaf_reg": 12,
    "random_strength": 1.2,
    "bootstrap_type": "Bernoulli",
    "task_type": "GPU",
    "devices": "0",
    "verbose": False
}

xgb_params = {
    "n_estimators": 10000,
    "learning_rate": 0.01,
    "max_depth": 2,
    "subsample": 0.65,
    "colsample_bytree": 0.8,
    "reg_lambda": 12,
    "objective": "binary:logistic",
    "eval_metric": "auc",
    "tree_method": "hist",
    "predictor": "gpu_predictor",
    "verbosity": 0
}

rf_params = {
    "n_estimators": 400,
    "max_depth": 6,
    "min_samples_leaf": 50,
    "class_weight": "balanced",
    "n_jobs": -1,
    "random_state": 42
}

In [9]:
# 5-FOLD GPU TRAINING + RANK STACK
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_rank = np.zeros(len(X_train))
test_rank = np.zeros(len(X_test))

for fold,(tr,val) in enumerate(skf.split(X_train, y)):
    print(f"Fold {fold+1}")

    cb = CatBoostClassifier(**cat_params, random_seed=42+fold)
    cb.fit(X_train.iloc[tr], y[tr])
    val_cb = cb.predict_proba(X_train.iloc[val])[:,1]
    te_cb  = cb.predict_proba(X_test)[:,1]

    xgb = XGBClassifier(**xgb_params, random_state=42+fold)
    xgb.fit(X_train.iloc[tr], y[tr])
    val_xgb = xgb.predict_proba(X_train.iloc[val])[:,1]
    te_xgb  = xgb.predict_proba(X_test)[:,1]

    val_rank = (rankdata(val_cb) + rankdata(val_xgb)) / 2 / len(val_cb)
    te_rank  = (rankdata(te_cb)  + rankdata(te_xgb))  / 2 / len(te_cb)

    oof_rank[val] = val_rank
    test_rank += te_rank / skf.n_splits

    print(" Rank AUC:", roc_auc_score(y[val], val_rank))
    gc.collect()

Fold 1


Default metric period is 5 because AUC is/are not implemented for GPU


 Rank AUC: 0.9560632087187979
Fold 2


Default metric period is 5 because AUC is/are not implemented for GPU


 Rank AUC: 0.9547698493432817
Fold 3


...

 Rank AUC: 0.9555132494930322
Fold 4


Default metric period is 5 because AUC is/are not implemented for GPU


 Rank AUC: 0.9552722179223394
Fold 5


Default metric period is 5 because AUC is/are not implemented for GPU


 Rank AUC: 0.9558706983581478


In [10]:
# RESIDUAL RANDOM FOREST
rf = RandomForestClassifier(**rf_params)
rf.fit(oof_rank.reshape(-1,1), y)

final_test = (
    0.85 * test_rank +
    0.15 * rf.predict_proba(test_rank.reshape(-1,1))[:,1]
)

In [11]:
# SUBMISSION
submission = pd.DataFrame({
    "id": test_ids,
    TARGET: final_test
})
submission.to_csv("submission.csv", index=False)

print("\nGPU submission saved: submission.csv")


GPU submission saved: submission.csv
